In [1]:
import napari
import numpy as np
import tifffile as tiff
import zarr
import dask.array as da
from dask.diagnostics import ProgressBar
import pandas as pd
import xml.etree.ElementTree as ET
import seaborn as sns
import matplotlib.pyplot as plt
import os

In [ ]:
def print_ome_tiff_metadata(image_path: str, print_full_xml: bool = False):
    """
    Print metadata structure of an OME-TIFF file without loading image data into RAM.
    
    Parameters:
    -----------
    image_path : str
        Path to the OME-TIFF file
    print_full_xml : bool, optional
        If True, prints the full OME-XML metadata (can be very long)
    """
    
    with tiff.TiffFile(image_path) as tif:
        print("=" * 60)
        print("OME-TIFF Metadata Structure")
        print("=" * 60)
        
        # Basic file info
        print(f"\nFile: {image_path}")
        print(f"Number of series: {len(tif.series)}")
        print(f"Number of pages: {len(tif.pages)}")
        print(f"Is OME-TIFF: {tif.is_ome}")
        
        # Series information
        for idx, series in enumerate(tif.series):
            print(f"\nSeries {idx}:")
            print(f"  Shape: {series.shape}")
            print(f"  Dtype: {series.dtype}")
            print(f"  Axes: {series.axes}")
            print(f"  Number of pages: {len(series)}")
        
        # OME-XML metadata
        if hasattr(tif, 'ome_metadata') and tif.ome_metadata:
            if print_full_xml:
                import xml.dom.minidom as minidom
                xml_str = minidom.parseString(tif.ome_metadata)
                print("\n" + "=" * 60)
                print("Full OME-XML Metadata:")
                print("=" * 60)
                print(xml_str.toprettyxml(indent="  "))
            else:
                print("\nOME-XML Metadata (first 1000 chars):")
                print(tif.ome_metadata[:1000])
                print("\n... (use print_full_xml=True to see full XML)")
        
        # ImageJ metadata
        if hasattr(tif, 'imagej_metadata') and tif.imagej_metadata:
            print("\nImageJ Metadata:")
            for key, value in tif.imagej_metadata.items():
                print(f"  {key}: {value}")
        
        # Page-level metadata (first page as example)
        if len(tif.pages) > 0:
            print("\nFirst Page Tags (example):")
            page = tif.pages[0]
            print(f"  ImageWidth: {page.imagewidth}")
            print(f"  ImageLength: {page.imagelength}")
            print(f"  BitsPerSample: {page.bitspersample}")
            print(f"  SamplesPerPixel: {page.samplesperpixel}")
            print(f"  Compression: {page.compression}")
        
        print("\n" + "=" * 60)

# Usage:
# print_ome_tiff_metadata("/path/to/your/file.ome.tif")
# print_ome_tiff_metadata("/path/to/your/file.ome.tif", print_full_xml=True)


In [4]:
path = '/Users/lukashat/sds_mount/sds/sd22c003/phenotyping_benchmark/datasets/cycif_ovca/raw_images/multistack_tiffs/S014_IDS.ome.tif'

In [8]:

with tiff.TiffFile(path) as tif:
    # Print raw OME-XML metadata
    print("=== OME-XML ===")
    print(tif.ome_metadata[:3000])

    # Parse channel names via ome_metadata structured object
    print("\n=== Channel names from series ===")
    series = tif.series[0]
    for i, level in enumerate(series.levels):
        print(f"  Level {i}: shape={level.shape}, axes={level.axes}")

    # # Try structured ome metadata
    # print("\n=== Structured OME ===")
    # ome = tiff.tifffile.OmeXml(tif.ome_metadata)  # may not exist in older tifffile
    # for img in ome.images:
    #     print(f"  Image: {img.name}")
    #     for ch in img.pixels.channels:
    #         print(f"    Channel: name={ch.name}, fluor={ch.fluor}")

=== OME-XML ===
<?xml version="1.0" encoding="UTF-8"?><OME xmlns="http://www.openmicroscopy.org/Schemas/OME/2016-06" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.openmicroscopy.org/Schemas/OME/2016-06 http://www.openmicroscopy.org/Schemas/OME/2016-06/ome.xsd" UUID="urn:uuid:68b33b95-790e-11ef-94ab-a002a566e9e5" Creator="tifffile.py 2024.8.30"><Image ID="Image:0" Name="Image0"><Pixels ID="Pixels:0" DimensionOrder="XYCZT" Type="uint16" SizeX="40140" SizeY="25713" SizeC="32" SizeZ="1" SizeT="1"><Channel ID="Channel:0:0" SamplesPerPixel="1"><LightPath/></Channel><Channel ID="Channel:0:1" SamplesPerPixel="1"><LightPath/></Channel><Channel ID="Channel:0:2" SamplesPerPixel="1"><LightPath/></Channel><Channel ID="Channel:0:3" SamplesPerPixel="1"><LightPath/></Channel><Channel ID="Channel:0:4" SamplesPerPixel="1"><LightPath/></Channel><Channel ID="Channel:0:5" SamplesPerPixel="1"><LightPath/></Channel><Channel ID="Channel:0:6" SamplesPerPixel="1"><LightPat

In [ ]:
# def open_image_napari(image_path, markers):
#     # create a colormap list with the same length as markers - ['cyan', 'green', 'yellow', 'red', 'magenta', 'orange', 'blue' ] repeat until reach len(markers)
#     colormap = ['cyan', 'green', 'yellow', 'red', 'magenta', 'orange', 'blue']
#     colormap = colormap * (len(markers) // len(colormap) + 1)

#     store = tiff.imread(image_path, aszarr=True)
#     cache = zarr.LRUStoreCache(store, max_size=2**30)
#     zobj = zarr.open(cache, mode='r')
#     data = [
#         zobj[int(dataset['path'])]
#         for dataset in zobj.attrs['multiscales'][0]['datasets']
#     ]
#     data = [da.from_zarr(z) for z in data]
#     n_channels = data[0].shape[0]
#     for z in data:
#         print(z)
#     viewer = napari.Viewer()
#     # Add each channel as a separate image layer
#     for i in range(n_channels):
#         channel_data = [data_layer[i] for data_layer in data]  # Extract the i-th channel
#         viewer.add_image(
#             channel_data,
#             name=markers.marker_name[i],
#             rgb=False,
#             contrast_limits=[0, 65535],
#             blending='additive',
#             colormap=colormap[i],
#         )
#     napari.run()
#     store.close()

#     return viewer

In [ ]:
sds_dir = "../../../../../../../../sds_mount/sds/sd22c003/phenotyping_benchmark/datasets/"

In [ ]:
ome_path = os.path.join(sds_dir, 'breast_fibro_IMC/raw_images/multistack_tiffs/20201013_LC_BC-Fibro_TBB075_s0_p8_r1_a1_ac.ome.tiff')

In [ ]:
def open_multiseries_image_napari(image_path:str, markers:list) -> napari.Viewer:
    colormap = ['cyan', 'green', 'yellow', 'red', 'magenta', 'orange', 'blue']
    colormap = colormap * (len(markers) // len(colormap) + 1)

    pyramid = []
    with tiff.TiffFile(image_path) as tif:
        for series in tif.series:
            store = series.aszarr()
            dask_array = da.from_zarr(store)
            pyramid.append(dask_array)
    n_channels = pyramid[0].shape[0]
    viewer = napari.Viewer()
    for i in range(n_channels):
        channel_data = [data_layer[i] for data_layer in pyramid]
        viewer.add_image(
            channel_data,
            name=markers[i],
            multiscale=True,
            contrast_limits=[0, 500],
            blending='additive',
            colormap=colormap[i],
        )
    napari.run()
    store.close()

    return viewer
    

In [ ]:
def open_multiseries_image_napari(image_path:str) -> napari.Viewer:
    colormap = ['cyan', 'green', 'yellow', 'red', 'magenta', 'orange', 'blue']
   

    pyramid = []
    with tiff.TiffFile(image_path) as tif:
        for series in tif.series:
            store = series.aszarr()
            dask_array = da.from_zarr(store)
            pyramid.append(dask_array)
    n_channels = pyramid[0].shape[0]
    colormap = colormap * (n_channels // len(colormap) + 1)
    viewer = napari.Viewer()
    for i in range(n_channels):
        channel_data = [data_layer[i] for data_layer in pyramid]
        viewer.add_image(
            channel_data,
            name=f'Channel {i+1}',
            multiscale=True,
            contrast_limits=[0, 65535],
            blending='additive',
            colormap=colormap[i],
        )
    napari.run()
    store.close()

    return viewer
    

In [ ]:
with open(os.path.join(sds_dir, 'breast_fibro_IMC/markers.txt')) as f:
    channel_names = [line.strip() for line in f.readlines()]

In [ ]:
open_multiseries_image_napari(ome_path, channel_names)

In [ ]:
print_ome_tiff_metadata(ome_path, print_full_xml=True)

In [ ]:
store = tiff.imread(image_path, aszarr=True)
cache = zarr.LRUStoreCache(store, max_size=2**30)
zobj = zarr.open(cache, mode='r')